# Sprint 5: Simulación de Impacto Económico (ROI)

Traducimos las métricas a valor de negocio.

In [1]:
import sys
sys.path.append("../src")
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/processed/features_data.csv")
X = df.drop(columns=['attrition'])
y = df['attrition']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe = joblib.load("../models/final_model.pkl")
probs = pipe.predict_proba(X_test)[:, 1]

cost_retention = 150
benefit_tp = 1000

df_val = pd.DataFrame({'real': y_test, 'prob': probs})
# Supongamos que alertamos si prob >= 0.35
df_val['pred'] = (df_val['prob'] >= 0.35).astype(int)

tp = ((df_val['pred'] == 1) & (df_val['real'] == 1)).sum()
fp = ((df_val['pred'] == 1) & (df_val['real'] == 0)).sum()

net_savings = (tp * benefit_tp) - ((tp + fp) * cost_retention)
print(f"Verdaderos Positivos (Fugas detectadas): {tp}")
print(f"Falsos Positivos (Dinero gastado innecesariamente): {fp}")
print(f"Ahorro Neto Estimado en el Test Set: ${net_savings:,.2f}")

Verdaderos Positivos (Fugas detectadas): 116
Falsos Positivos (Dinero gastado innecesariamente): 515
Ahorro Neto Estimado en el Test Set: $21,350.00
